In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

/Users/ahirverma/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
# We will use state data instead of regional data 
DATA_DIR = Path.cwd() / "data"
DATA_DIR

PosixPath('/Users/ahirverma/Downloads/Freight-Forecasting-main/data')

## Load in employment data

In [3]:
cbp_df = pd.read_csv(DATA_DIR / "cbp22st.txt", dtype={"fipstate": str, "naics": str})
cbp_df = cbp_df[cbp_df["lfo"] == "-"]
cbp_df["code"] = cbp_df["naics"].str.rstrip("/-")          # "311///" -> "311"
emp_df = cbp_df.pivot_table(index="fipstate", columns="code", values="emp", aggfunc="sum").fillna(0)
emp_df

code,,11,113,1131,11311,113110,1132,11321,113210,1133,...,813910,81392,813920,81393,813930,81394,813940,81399,813990,99
fipstate,,,,,,,,,,,,,,,,,,,,,
01,1777657.0,5501.0,3785.0,259.0,259.0,259.0,43.0,43.0,43.0,3483.0,...,1447.0,351.0,351.0,1025.0,1025.0,48.0,48.0,344.0,344.0,139.0
02,257462.0,675.0,157.0,0.0,0.0,0.0,0.0,0.0,0.0,150.0,...,226.0,157.0,157.0,308.0,308.0,19.0,19.0,92.0,92.0,14.0
04,2787701.0,1941.0,122.0,0.0,0.0,0.0,0.0,0.0,0.0,73.0,...,1083.0,1265.0,1265.0,899.0,899.0,308.0,308.0,4222.0,4222.0,167.0
05,1089044.0,4780.0,2200.0,247.0,247.0,247.0,23.0,23.0,23.0,1930.0,...,797.0,166.0,166.0,430.0,430.0,41.0,41.0,838.0,838.0,97.0
06,16032440.0,33361.0,2126.0,276.0,276.0,276.0,102.0,102.0,102.0,1748.0,...,8336.0,5136.0,5136.0,15018.0,15018.0,2109.0,2109.0,12803.0,12803.0,1577.0
08,2481196.0,1754.0,104.0,26.0,26.0,26.0,0.0,0.0,0.0,78.0,...,1737.0,1799.0,1799.0,1439.0,1439.0,198.0,198.0,3388.0,3388.0,195.0
09,1508634.0,325.0,33.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,...,1721.0,640.0,640.0,2028.0,2028.0,74.0,74.0,779.0,779.0,75.0
10,424303.0,136.0,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,340.0,87.0,87.0,339.0,339.0,31.0,31.0,131.0,131.0,208.0
11,527355.0,15.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,11796.0,9919.0,9919.0,7039.0,7039.0,2602.0,2602.0,865.0,865.0,13.0


In [4]:
# codes come from: https://www2.census.gov/programs-surveys/cbp/technical-documentation/reference/naics-descriptions/naics2017.txt

sector_map = {"emp_manufacturing": ["31"],                  # manufacturing (31-33)
              "emp_mining": ["21"],                         # mining, oil and gas extraction
              "emp_wholesale_warehousing": ["42", "493"]}   # wholesale trade + warehousing and storage
E_prod = pd.DataFrame({col: emp_df[codes].sum(axis=1) for col, codes in sector_map.items()})
E_prod

,emp_manufacturing,emp_mining,emp_wholesale_warehousing
fipstate,,,
01,267658.0,4902.0,94834.0
02,10824.0,9214.0,9025.0
04,171133.0,12105.0,129973.0
05,161239.0,3076.0,60097.0
06,1160856.0,15982.0,989048.0
08,135443.0,19012.0,119528.0
09,151851.0,527.0,77856.0
10,29767.0,271.0,26782.0
11,1014.0,0.0,3433.0


## Load metadata about states, i.e., FIPS codes and commodity codes

In [5]:
metadata = pd.read_excel(DATA_DIR / "state" / "FAF6_Metadata.xlsx", ["State", "Commodity (SCTG2)"])

In [6]:
states = metadata["State"]
state_to_code = dict(zip(states["Description"], states["Numeric Label"]))
state_to_code

{'Alabama': 1,
 'Alaska': 2,
 'Arizona': 4,
 'Arkansas': 5,
 'California': 6,
 'Colorado': 8,
 'Connecticut': 9,
 'Delaware': 10,
 'Washington DC': 11,
 'Florida': 12,
 'Georgia': 13,
 'Hawaii': 15,
 'Idaho': 16,
 'Illinois': 17,
 'Indiana': 18,
 'Iowa': 19,
 'Kansas': 20,
 'Kentucky': 21,
 'Louisiana': 22,
 'Maine': 23,
 'Maryland': 24,
 'Massachusetts': 25,
 'Michigan': 26,
 'Minnesota': 27,
 'Mississippi': 28,
 'Missouri': 29,
 'Montana': 30,
 'Nebraska': 31,
 'Nevada': 32,
 'New Hampshire': 33,
 'New Jersey': 34,
 'New Mexico': 35,
 'New York': 36,
 'North Carolina': 37,
 'North Dakota': 38,
 'Ohio': 39,
 'Oklahoma': 40,
 'Oregon': 41,
 'Pennsylvania': 42,
 'Rhode Island': 44,
 'South Carolina': 45,
 'South Dakota': 46,
 'Tennessee': 47,
 'Texas': 48,
 'Utah': 49,
 'Vermont': 50,
 'Virginia': 51,
 'Washington': 53,
 'West Virginia': 54,
 'Wisconsin': 55,
 'Wyoming': 56}

In [7]:
commodities = metadata["Commodity (SCTG2)"]
code_to_commodity = dict(zip(commodities["Numeric Label"], commodities["Description"]))
code_to_commodity

{1: 'Live animals/fish',
 2: 'Cereal grains',
 3: 'Other ag prods.',
 4: 'Animal feed',
 5: 'Meat/seafood',
 6: 'Milled grain prods.',
 7: 'Other foodstuffs',
 8: 'Alcoholic beverages',
 9: 'Tobacco prods.',
 10: 'Building stone',
 11: 'Natural sands',
 12: 'Gravel',
 13: 'Nonmetallic minerals',
 14: 'Metallic ores',
 15: 'Coal',
 16: 'Crude petroleum',
 17: 'Gasoline',
 18: 'Fuel oils',
 19: 'Natural gas and other fossil products',
 20: 'Basic chemicals',
 21: 'Pharmaceuticals',
 22: 'Fertilizers',
 23: 'Chemical prods.',
 24: 'Plastics/rubber',
 25: 'Logs',
 26: 'Wood prods.',
 27: 'Newsprint/paper',
 28: 'Paper articles',
 29: 'Printed prods.',
 30: 'Textiles/leather',
 31: 'Nonmetal min. prods.',
 32: 'Base metals',
 33: 'Articles-base metal',
 34: 'Machinery',
 35: 'Electronics',
 36: 'Motorized vehicles',
 37: 'Transport equip.',
 38: 'Precision instruments',
 39: 'Furniture',
 40: 'Misc. mfg. prods.',
 41: 'Waste/scrap',
 43: 'Mixed freight'}

## Load in GDP and per capita personal income values

In [8]:
gdp_income_df = pd.read_csv(
    DATA_DIR / "GDP_Income_Data.csv",
    usecols=["GeoFIPS", "GeoName", "Description", "2022"], 
    skiprows=[0, 1, 2, 212, 213, 214, 215, 216, 217]
)
gdp_income_df = gdp_income_df.dropna()
gdp_income_df = gdp_income_df[gdp_income_df["GeoName"] != "United States"]
gdp_income_df["GeoName"] = gdp_income_df["GeoName"].replace(
    "District of Columbia", "Washington DC"
)
gdp_income_df.insert(
    gdp_income_df.columns.get_loc("GeoName") + 1,
    "StateCode",
    gdp_income_df["GeoName"].map(state_to_code),
)
gdp_income_df

,GeoFIPS,GeoName,StateCode,Description,2022
5,1000,Alabama,1,Real GDP (millions of chained 2017 dollars) 1,240210.5
7,1000,Alabama,1,Real per capita personal income 4,50649.0
9,2000,Alaska,2,Real GDP (millions of chained 2017 dollars) 1,50384.5
11,2000,Alaska,2,Real per capita personal income 4,58413.0
13,4000,Arizona,4,Real GDP (millions of chained 2017 dollars) 1,414460.2
...,...,...,...,...,...
199,54000,West Virginia,54,Real per capita personal income 4,48517.0
201,55000,Wisconsin,55,Real GDP (millions of chained 2017 dollars) 1,342786.9
203,55000,Wisconsin,55,Real per capita personal income 4,57706.0
205,56000,Wyoming,56,Real GDP (millions of chained 2017 dollars) 1,37338.1


In [9]:
gdp_income_df.head(30)

,GeoFIPS,GeoName,StateCode,Description,2022
5,1000,Alabama,1,Real GDP (millions of chained 2017 dollars) 1,240210.5
7,1000,Alabama,1,Real per capita personal income 4,50649.0
9,2000,Alaska,2,Real GDP (millions of chained 2017 dollars) 1,50384.5
11,2000,Alaska,2,Real per capita personal income 4,58413.0
13,4000,Arizona,4,Real GDP (millions of chained 2017 dollars) 1,414460.2
15,4000,Arizona,4,Real per capita personal income 4,50935.0
17,5000,Arkansas,5,Real GDP (millions of chained 2017 dollars) 1,137623.6
19,5000,Arkansas,5,Real per capita personal income 4,53291.0
21,6000,California,6,Real GDP (millions of chained 2017 dollars) 1,3161153.4
23,6000,California,6,Real per capita personal income 4,59126.0


## Load in census data

In [10]:
census_2022_df = pd.read_excel(
    DATA_DIR / "NST-EST2022-POP.xlsx",
    skiprows=[0, 1, 2, 3, 4, 5, 6, 7, 60, 61, 62, 63, 64, 65, 66]
)
census_2022_df.columns = ["State", "Base", "2020", "2021", "2022"]
census_2022_df["State"] = census_2022_df["State"].str.lstrip(".")
census_2022_df["State"] = census_2022_df["State"].replace(
    "District of Columbia", "Washington DC"
)
census_2022_df.insert(
    census_2022_df.columns.get_loc("State") + 1,
    "StateCode",
    census_2022_df["State"].map(state_to_code),
)
census_2022_df = census_2022_df.drop(["Base", "2020", "2021"], axis=1)
census_2022_df

,State,StateCode,2022
0,Alabama,1,5074296
1,Alaska,2,733583
2,Arizona,4,7359197
3,Arkansas,5,3045637
4,California,6,39029342
5,Colorado,8,5839926
6,Connecticut,9,3626205
7,Delaware,10,1018396
8,Washington DC,11,671803
9,Florida,12,22244823


## Construct datasets for forecasting $P_{ic}$ and $A_{jc}$

In [11]:
# Align everything on integer state codes, in the same order used for the OD matrices below
state_index = pd.Index(list(state_to_code.values()), name="StateCode")
E = E_prod.copy()
E.index = E.index.astype(int)
E = E.reindex(state_index)

def gdp_income_series(keyword):
    rows = gdp_income_df[gdp_income_df["Description"].str.contains(keyword, regex=False)]
    return rows.set_index("StateCode")["2022"].astype(float).reindex(state_index)

real_gdp = gdp_income_series("Real GDP")
real_income = gdp_income_series("Real per capita personal income")
population = census_2022_df.set_index("StateCode")["2022"].astype(float).reindex(state_index)

X_P = pd.DataFrame({
    "emp_manufacturing_mining": E["emp_manufacturing"] + E["emp_mining"],
    "emp_wholesale_warehousing": E["emp_wholesale_warehousing"],
    "real_gdp": real_gdp,
}, index=state_index)

X_A = pd.DataFrame({
    "population": population,
    "real_per_capita_income": real_income,
    "emp_manufacturing": E["emp_manufacturing"],
}, index=state_index)

assert X_P.shape == (51, 3) and X_A.shape == (51, 3)
assert not X_P.isna().any().any() and not X_A.isna().any().any()

X_P

,emp_manufacturing_mining,emp_wholesale_warehousing,real_gdp
StateCode,,,
1,272560.0,94834.0,240210.5
2,20038.0,9025.0,50384.5
4,183238.0,129973.0,414460.2
5,164315.0,60097.0,137623.6
6,1176838.0,989048.0,3161153.4
8,154455.0,119528.0,422368.4
9,152378.0,77856.0,280823.1
10,30038.0,26782.0,85292.7
11,1014.0,3433.0,145227.3


In [12]:
X_A

,population,real_per_capita_income,emp_manufacturing
StateCode,,,
1,5074296.0,50649.0,267658.0
2,733583.0,58413.0,10824.0
4,7359197.0,50935.0,171133.0
5,3045637.0,53291.0,161239.0
6,39029342.0,59126.0,1160856.0
8,5839926.0,64668.0,135443.0
9,3626205.0,69250.0,151851.0
10,1018396.0,56371.0,29767.0
11,671803.0,77241.0,1014.0


## Load in FAF6 data

In [13]:
df = pd.read_csv(DATA_DIR / "state" / "FAF6.0_State.csv")
# Filter to domestic flows
# TODO: Decide whether we want to do this
# df = df[df["trade_type"] == 1]
df

,fr_orig,dms_origst,dms_destst,fr_dest,fr_inmode,dms_mode,fr_outmode,sctg2,trade_type,tons_2022,value_2022
0,NaN,1,1,NaN,NaN,1,NaN,1,1,924.475048,1800.959032
1,NaN,1,5,NaN,NaN,1,NaN,1,1,88.641677,170.193456
2,NaN,1,12,NaN,NaN,1,NaN,1,1,28.002798,53.972010
3,NaN,1,13,NaN,NaN,1,NaN,1,1,150.320804,292.338206
4,NaN,1,20,NaN,NaN,1,NaN,1,1,9.563894,30.829675
...,...,...,...,...,...,...,...,...,...,...,...
599524,NaN,53,53,808.0,NaN,1,3.0,43,3,174.640924,706.122872
599525,NaN,53,53,808.0,NaN,1,4.0,43,3,0.000042,0.001840
599526,NaN,55,6,808.0,NaN,5,3.0,43,3,0.001996,0.007644
599527,NaN,55,45,808.0,NaN,2,3.0,43,3,0.000174,0.001099


## Create OD matrices for each commodity

In [14]:
state_codes = list(state_to_code.values())
commodity_codes = list(code_to_commodity.keys())

totals = (
    df.groupby(["sctg2", "dms_origst", "dms_destst"])["tons_2022"]
    .sum()
    .reindex(
        pd.MultiIndex.from_product(
            [commodity_codes, state_codes, state_codes],
            names=["sctg2", "dms_origst", "dms_destst"],
        ),
        fill_value=0,
    )
)

od_matrices = {
    code: totals.loc[code]
    .unstack("dms_destst")
    .reindex(index=state_codes, columns=state_codes)
    for code in commodity_codes
}

# This is the OD matrix corresponding to commodity "1" (Live animals/fish)
od_matrices[1]

dms_destst,1,2,4,5,6,8,9,10,11,12,...,46,47,48,49,50,51,53,54,55,56
dms_origst,,,,,,,,,,,,,,,,,,,,,
1,941.067092,0.000000,0.000000,88.641677,0.000000,0.000000,0.000000,0.000000,0.000000,28.031736,...,0.000000,53.476749,51.355797,0.000000,0.000000,0.157649,0.167137,0.000000,0.000000,0.000000
2,0.000000,2484.783180,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000033,...,0.000000,0.000000,0.004790,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.000000,0.000119,227.349691,0.000000,105.312465,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,1.892019,0.000000,0.000000,0.000000,0.033376,0.000000,0.000000,0.000000
5,135.295282,0.000000,0.000000,696.406032,0.000110,0.000000,0.000000,0.000000,0.000000,0.021470,...,0.000000,98.991316,116.064751,0.000000,0.000000,0.000000,0.256059,0.000000,0.000000,0.000000
6,0.000000,0.000139,19.537216,0.000000,4858.642969,0.020669,0.000000,0.000000,0.008203,0.032236,...,0.155812,0.092433,0.310786,0.000259,0.000000,0.000428,1.367144,0.000000,0.000038,0.001984
8,0.000000,0.000000,0.000000,0.000000,0.001653,1396.455311,0.000000,0.000000,0.000000,0.001655,...,0.000000,0.100635,0.028522,0.000163,0.000000,0.048265,0.002743,0.000000,0.000000,0.000163
9,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,23.185287,0.000000,0.000000,0.000000,...,0.000000,0.000570,0.013547,0.000000,0.000000,0.858562,0.000000,0.000000,0.000000,0.000000
10,0.000000,0.000000,0.000000,0.000000,0.000001,0.000000,0.157876,15.489616,0.000000,0.000250,...,0.000000,0.000000,0.000000,0.000000,0.000000,2.331477,0.000000,0.000000,0.000000,0.000000
11,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000212,0.000000,...,0.000000,0.000000,0.000362,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


The FAF6 data gives us an OD matrix for each commodity. That is, we have $T_{ijc}$, where $i$ is the origin state, $j$ is the destination state, and $c$ is the commodity type. This gives us the base number of "productions" and "attractions," i.e., $P_{ic} = \sum_j T_{ijc}$ and $A_{jc} = \sum_i T_{ijc}$. We can regress over these quantities for each state and commodity. For productions, we have
\begin{align*}
P_{ic} = \alpha_c + \beta_c x_{i1} + \gamma_c x_{i2} + \delta_c x_{i3}
\end{align*}
where $x_{i1}$ is the goods-producing employment (manufacturing and mining) in state $i$, $x_{i2}$ is the wholesale and warehousing employment in state $i$, and $x_{i3}$ is the real state GDP of state $i$. Likewise, for attractions we have
\begin{align*}
A_{jc} = \tilde{\alpha}_c + \tilde{\beta}_c \tilde{x}_{i1} + \tilde{\gamma}_c \tilde{x}_{i2} + \tilde{\delta}_c \tilde{x}_{i3}
\end{align*}
where $\tilde{x}_{i1}$ is the population of state $j$, $\tilde{x}_{i2}$ is the real per capita personal income in state $j$, and $\tilde{x}_{i3}$ is the manufacturing employment in state $j$. These are the more readily available quantities I was able to find from official government sources, so I decided to go with them, and I think it checks out. The data matrices $X_P$ and $X_A$ are already created in an earlier code cell, so all that is left to do is regress over productions and attractions for each state and commodity. Note that we can maybe do some transformations like log-log or inverse sinh and compare their $R^2$ values. 

In [15]:
import statsmodels.api as sm

## Compute base-year productions and attractions

In [17]:
# Rows of each OD matrix are origins i, columns are destinations j
#   P_ic = sum_j T_ijc  -> row sums
#   A_jc = sum_i T_ijc  -> column sums
P = pd.DataFrame({c: od.sum(axis=1) for c, od in od_matrices.items()}).reindex(state_index)
A = pd.DataFrame({c: od.sum(axis=0) for c, od in od_matrices.items()}).reindex(state_index)
P.index.name = A.index.name = "StateCode"
P

,1,2,3,4,5,6,7,8,9,10,...,33,34,35,36,37,38,39,40,41,43
StateCode,,,,,,,,,,,,,,,,,,,,,
1,1487.016566,1871.520282,2661.116522,12234.686292,3350.073561,683.749431,5717.905396,675.783384,2.972910,53.819695,...,3667.657203,2258.273098,1051.937738,6536.017039,220.224769,152.692408,1221.173943,3792.416226,13462.579691,4247.712728
2,2484.788003,6.482852,69.619617,137.125182,1214.845823,72.810137,307.961016,243.263957,0.424487,0.931309,...,68.315783,256.165607,93.978290,25.544970,2.508015,11.173363,109.090615,121.443148,699.677004,250.065664
4,344.078424,1001.375050,6166.832294,6209.590008,429.325301,1061.032521,6121.742735,3661.639181,62.987699,164.912611,...,1199.490787,1337.771414,1352.598981,2333.085333,173.143619,636.654752,1136.240563,1971.144032,11187.569503,13146.852896
5,1397.566053,11521.204117,9093.796241,5726.232795,4001.374596,2963.283615,3379.964480,758.768422,32.688284,23.672991,...,2480.037967,956.522429,737.995464,1369.045634,65.650991,31.126620,486.202255,2638.736085,5809.016653,5444.495953
6,4921.984625,5208.008988,59487.781124,35538.588513,10195.280571,13867.440439,179896.956392,19878.966712,319.779128,486.527749,...,15968.676699,12278.402401,33288.516574,41482.332744,902.515223,2852.370235,12930.033145,16575.360136,56394.282793,44953.801522
8,1710.880051,14156.399276,3356.380002,6383.142289,2756.640861,1605.606862,10810.453315,2298.205563,3.995010,752.153045,...,1955.617464,1024.244435,1065.192793,1043.991831,215.361629,237.950948,1243.393457,1986.082210,10022.013731,5281.481447
9,64.070063,30.084399,2810.362329,896.072238,308.199310,1333.665854,1508.951415,1011.765805,62.508390,315.501037,...,999.492978,895.246230,2305.528868,308.703517,57.671476,91.658745,657.716106,730.299898,8181.009359,4949.946046
10,32.933140,1354.863985,2895.099488,2225.769173,1156.391722,26.562681,2754.829762,501.352335,1.447800,4.707903,...,391.043829,115.839237,1144.715431,434.898727,46.747058,53.573446,156.473915,906.750016,2405.734082,3874.583780
11,0.001016,0.014237,41.130623,79.027755,28.656402,25.418277,38.421951,178.648442,0.206953,4.898974,...,14.661726,18.457196,100.860494,46.385514,56.182400,3.611697,166.024119,121.258059,846.086732,187.683347


In [18]:
A

,1,2,3,4,5,6,7,8,9,10,...,33,34,35,36,37,38,39,40,41,43
StateCode,,,,,,,,,,,,,,,,,,,,,
1,1511.510730,4948.637353,3768.477191,10155.791450,1721.837471,791.430525,8064.455753,1570.180144,28.833830,89.426933,...,3021.910169,2387.693181,1818.138077,5755.935564,175.261391,164.193624,1256.129287,4198.180414,17686.996866,5393.926971
2,2484.796669,7.068113,104.896216,144.886772,834.392890,75.189025,402.244539,269.159562,1.495746,0.931309,...,148.810768,336.562773,115.483663,203.969479,9.081204,20.576165,121.001320,164.215047,623.396822,421.634694
4,291.875534,1309.970948,5473.740988,8097.069885,1242.151390,1470.655240,13813.662249,3234.915304,70.026111,171.990955,...,1900.553553,1544.077248,2281.084167,3021.250146,153.946287,392.190749,1725.500031,2822.549427,10834.586043,12602.165387
5,1402.544395,6296.142804,5896.561252,8477.200386,3332.658938,1440.344146,4503.847818,995.459981,33.274858,30.265260,...,1365.519542,1055.361843,697.162076,1265.225344,95.609287,85.144959,657.994844,1808.193278,8991.001133,5646.040970
6,5106.702789,17820.995569,62529.110726,44135.663846,15260.019131,13813.478068,146026.675818,16560.675745,284.301962,390.304810,...,14323.665469,10783.692497,26658.643705,43775.116636,913.043918,2690.242064,10278.878536,14233.097542,57876.956729,40546.769602
8,2118.012291,15378.873726,2758.353563,6368.143652,1532.013067,1482.050237,12105.116618,2199.221827,24.455521,755.879581,...,1524.281483,1677.964671,1782.963334,1366.879231,272.770073,272.211835,1485.018719,2280.024620,9647.756774,5801.810162
9,136.568395,59.869374,1713.621585,977.966129,837.026433,859.866314,4096.244161,1451.796370,47.677937,350.829545,...,1021.486071,1027.192338,2030.912364,826.608286,53.947774,164.911971,850.951263,923.163669,4167.188842,6494.894086
10,15.582664,1240.219429,1836.718964,2389.335008,348.322251,91.795703,1570.145500,763.840780,5.297562,24.047623,...,351.308088,179.731013,417.135116,918.674567,276.222569,99.234404,243.290324,302.690909,1794.304928,1692.317853
11,1.044014,240.348109,221.202521,111.447182,70.551694,35.622993,758.417228,286.780989,2.318265,1.020485,...,129.130035,35.610931,153.939159,25.860655,1.216260,8.824588,230.286286,122.381971,0.782115,454.538434


## Define functional forms and regression routine
Next we fit each commodity's model in three forms: linear, log-log, and inverse hyperbolic sine (asinh). Log-log discards states with zero tons, whereas asinh behaves like log for large values but retains zeros. Since $R^2$ on transformed scales can't be compared directly, we also compute $R^2$ after transforming predictions back to tons.

In [19]:
TRANSFORMS = {
    "linear": (lambda v: v, lambda v: v),
    "log-log": (np.log, np.exp),
    "asinh": (np.arcsinh, np.sinh),
}

In [21]:
def fit_generation(Y, X, label):
    # Fit y_c = a_c + X b_c for every commodity c under each transform. Returns (models, summary)
    
    models, rows = {}, []
    
    for c in Y.columns:
        y = Y[c]
        
        for name, (f, f_inv) in TRANSFORMS.items():
            
            mask = (y > 0) & (X > 0).all(axis=1) if name == "log-log" else pd.Series(True, index=y.index)
            res = sm.OLS(f(y[mask]), sm.add_constant(f(X[mask]))).fit()
            models[(c, name)] = res
            
            # R^2 back on the original (tons) scale, so the three forms can be compared fairly
            y_hat = f_inv(res.fittedvalues)
            r2_tons = 1 - ((y[mask] - y_hat) ** 2).sum() / ((y[mask] - y[mask].mean()) ** 2).sum()
            
            rows.append({
                "model": label, "sctg2": c, "commodity": code_to_commodity[c], "transform": name,
                "n": int(res.nobs), "r2": res.rsquared, "adj_r2": res.rsquared_adj, "r2_tons": r2_tons,
                **{f"coef_{k}": v for k, v in res.params.items()},
                **{f"p_{k}": v for k, v in res.pvalues.items()},
            })
    return models, pd.DataFrame(rows)

## Regress productions $P_{ic}$ on $X_P$

In [22]:
P_models, P_summary = fit_generation(P, X_P, "productions")
P_summary

,model,sctg2,commodity,transform,n,r2,adj_r2,r2_tons,coef_const,coef_emp_manufacturing_mining,coef_emp_wholesale_warehousing,coef_real_gdp,p_const,p_emp_manufacturing_mining,p_emp_wholesale_warehousing,p_real_gdp
0,productions,1,Live animals/fish,linear,51,0.231071,0.181990,0.231071,792.713518,0.004728,0.004610,-0.001903,0.056129,0.107611,0.577176,0.353693
1,productions,1,Live animals/fish,log-log,51,0.702320,0.683319,0.022215,1.716639,1.439291,1.682711,-2.514344,0.580034,0.014371,0.110555,0.000583
2,productions,1,Live animals/fish,asinh,51,0.570153,0.542716,0.101152,0.192189,1.239681,0.386817,-0.994476,0.946489,0.017053,0.674715,0.105941
3,productions,2,Cereal grains,linear,51,0.146737,0.092274,0.146737,11341.445539,0.074990,0.044472,-0.040892,0.053752,0.072537,0.703879,0.161840
4,productions,2,Cereal grains,log-log,51,0.585641,0.559192,-0.034312,-0.277511,1.305889,2.561244,-2.921097,0.954469,0.148263,0.121386,0.008852
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121,productions,41,Waste/scrap,log-log,51,0.917862,0.912619,0.810384,-1.511713,-0.001110,0.644836,0.259448,0.048838,0.993587,0.013438,0.123313
122,productions,41,Waste/scrap,asinh,51,0.917862,0.912619,0.810384,-1.444598,-0.001110,0.644835,0.259448,0.070509,0.993587,0.013438,0.123312
123,productions,43,Mixed freight,linear,51,0.901357,0.895060,0.901357,630.866266,0.008279,0.065646,-0.006705,0.367955,0.100533,0.000026,0.060277
124,productions,43,Mixed freight,log-log,51,0.897985,0.891473,0.838533,-3.949856,0.220523,0.750973,0.107004,0.000406,0.253264,0.036286,0.643184


In [24]:
# Example: full regression output for one commodity and form
P_models[(43, "log-log")].summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                     43   R-squared:                       0.898
Model:                            OLS   Adj. R-squared:                  0.891
Method:                 Least Squares   F-statistic:                     137.9
Date:                Tue, 17 Nov 2026   Prob (F-statistic):           2.67e-23
Time:                        15:49:55   Log-Likelihood:                -29.521
No. Observations:                  51   AIC:                             67.04
Df Residuals:                      47   BIC:                             74.77
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
=============================================================================================
                                coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------
const                        -3.9499      1.037     -3.807      0.000      -6.037      -1.863
emp_manufacturing_mining      0.2205      0.191      1.157      0.253      -0.163       0.604
emp_wholesale_warehousing     0.7510      0.348      2.155      0.036       0.050       1.452
real_gdp                      0.1070      0.229      0.466      0.643      -0.355       0.569
==============================================================================
Omnibus:                        0.007   Durbin-Watson:                   1.720
Prob(Omnibus):                  0.997   Jarque-Bera (JB):                0.120
Skew:                          -0.025   Prob(JB):                        0.942
Kurtosis:                       2.768   Cond. No.                         358.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

## Regress attractions $A_{jc}$ on $X_A$

In [25]:
A_models, A_summary = fit_generation(A, X_A, "attractions")
A_summary

,model,sctg2,commodity,transform,n,r2,adj_r2,r2_tons,coef_const,coef_population,coef_real_per_capita_income,coef_emp_manufacturing,p_const,p_population,p_real_per_capita_income,p_emp_manufacturing
0,attractions,1,Live animals/fish,linear,51,0.257795,0.210421,0.257795,-1641.537765,-0.000055,0.041707,0.005974,0.511334,4.681640e-01,0.327355,1.496062e-02
1,attractions,1,Live animals/fish,log-log,51,0.515309,0.484371,0.200142,-19.909631,-0.505449,1.771662,1.260509,0.301689,1.659104e-01,0.304638,4.648604e-05
2,attractions,1,Live animals/fish,asinh,51,0.512231,0.481097,0.199976,-21.639132,-0.484665,1.824912,1.238401,0.288880,1.816307e-01,0.288250,5.617840e-05
3,attractions,2,Cereal grains,linear,51,0.183319,0.131190,0.183319,-39878.758750,-0.002081,0.878670,0.091593,0.235538,4.411661e-02,0.125837,5.740832e-03
4,attractions,2,Cereal grains,log-log,51,0.382371,0.342947,-0.033447,-71.974772,-0.592020,6.388546,1.624366,0.034201,3.457660e-01,0.035710,1.642621e-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121,attractions,41,Waste/scrap,log-log,51,0.820401,0.808937,0.695921,24.602539,0.138941,-2.680411,0.969983,0.039084,5.276555e-01,0.012973,8.191541e-07
122,attractions,41,Waste/scrap,asinh,51,0.826494,0.815420,0.710067,25.346691,0.171053,-2.598113,0.935889,0.038610,4.217856e-01,0.012743,8.496531e-07
123,attractions,43,Mixed freight,linear,51,0.899635,0.893229,0.899635,-1444.469934,0.001135,0.034189,0.006043,0.744334,4.605939e-11,0.649740,1.566354e-01
124,attractions,43,Mixed freight,log-log,51,0.954927,0.952050,0.876435,-17.973039,0.695364,1.103622,0.332240,0.000030,1.693503e-12,0.002693,5.462638e-07


In [26]:
A_models[(43, "log-log")].summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                     43   R-squared:                       0.955
Model:                            OLS   Adj. R-squared:                  0.952
Method:                 Least Squares   F-statistic:                     331.9
Date:                Tue, 17 Nov 2026   Prob (F-statistic):           1.27e-31
Time:                        15:50:38   Log-Likelihood:               0.098428
No. Observations:                  51   AIC:                             7.803
Df Residuals:                      47   BIC:                             15.53
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==========================================================================================
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                    -17.9730      3.890     -4.620      0.000     -25.799     -10.147
population                 0.6954      0.073      9.489      0.000       0.548       0.843
real_per_capita_income     1.1036      0.348      3.169      0.003       0.403       1.804
emp_manufacturing          0.3322      0.057      5.797      0.000       0.217       0.448
==============================================================================
Omnibus:                        0.822   Durbin-Watson:                   2.022
Prob(Omnibus):                  0.663   Jarque-Bera (JB):                0.431
Skew:                           0.221   Prob(JB):                        0.806
Kurtosis:                       3.090   Cond. No.                     2.46e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 2.46e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

## Compare functional forms

In [27]:
# Adjusted R^2 (transformed scale) and R^2 in tons, by commodity
r2_compare = pd.concat([P_summary, A_summary]).pivot_table(
    index=["model", "sctg2", "commodity"], columns="transform", values=["adj_r2", "r2_tons"]
).round(3)
r2_compare

adj_r2                r2_tons         \
transform                                asinh linear log-log   asinh linear   
model       sctg2 commodity                                                    
attractions 1     Live animals/fish      0.481  0.210   0.484   0.200  0.258   
            2     Cereal grains          0.343  0.131   0.343  -0.033  0.183   
            3     Other ag prods.        0.574  0.232   0.574   0.226  0.278   
            4     Animal feed            0.553  0.484   0.553   0.437  0.515   
            5     Meat/seafood           0.783  0.821   0.783   0.817  0.832   
...                                        ...    ...     ...     ...    ...   
productions 38    Precision instruments  0.881  0.813   0.881   0.771  0.824   
            39    Furniture              0.952  0.958   0.952   0.943  0.960   
            40    Misc. mfg. prods.      0.839  0.718   0.839   0.699  0.735   
            41    Waste/scrap            0.913  0.788   0.913   0.810  0.801   
            43    Mixed freight          0.891  0.895   0.891   0.839  0.901   

                                                 
transform                               log-log  
model       sctg2 commodity                      
attractions 1     Live animals/fish       0.200  
            2     Cereal grains          -0.033  
            3     Other ag prods.         0.226  
            4     Animal feed             0.437  
            5     Meat/seafood            0.817  
...                                         ...  
productions 38    Precision instruments   0.771  
            39    Furniture               0.943  
            40    Misc. mfg. prods.       0.699  
            41    Waste/scrap             0.810  
            43    Mixed freight           0.839  

[84 rows x 6 columns]

In [28]:
# Median fit across commodities for each model and form
pd.concat([P_summary, A_summary]).groupby(["model", "transform"])[["r2", "adj_r2", "r2_tons"]].median().round(3)

r2  adj_r2  r2_tons
model       transform                        
attractions asinh      0.779   0.765    0.625
            linear     0.679   0.659    0.679
            log-log    0.779   0.765    0.625
productions asinh      0.667   0.645    0.395
            linear     0.538   0.508    0.538
            log-log    0.691   0.671    0.393